# HTS Classifier

In [1]:
# Imports


## Load Dataset 
<br>

<footer> The original Hugging Face dataset is no longer publicly available.
The project was developed using a locally cached copy downloaded on June 2, 2026. </footer>

In [2]:
from src.data_loader import *

df = load_cross_dataset()
hts = load_hts_schedule(
    "data/hts_2026_revision_10_csv.csv"
)

In [3]:
df.head()

,messages
0,"[{'role': 'user', 'content': 'What is the HTS ..."
1,"[{'role': 'user', 'content': 'What is the HTS ..."
2,"[{'role': 'user', 'content': 'What is the HTS ..."
3,"[{'role': 'user', 'content': 'What is the HTS ..."
4,"[{'role': 'user', 'content': 'What is the HTS ..."


In [4]:
df.shape

(18254, 1)

In [5]:
df.columns

Index(['messages'], dtype='object')

In [6]:
# Look at one full example
df.iloc[0]["messages"]

array([{'role': 'user', 'content': 'What is the HTS US Code for a horizontal injection molding machine and molds for producing rubber parts?'},
       {'role': 'assistant', 'content': "HTS US Code -> 8477.10.9015; 8480.71.8045\nReasoning -> The horizontal injection molding machine is classified under HTS code 8477.10.9015, which pertains to machinery for working rubber or plastics or for the manufacture of products from these materials, specifically injection-molding machines used for processing rubber or other thermosetting materials. The machine's features, including a clamping force of 220 tons, powered mold height adjustment, a central ejector, a computerized control system, and a double brush discharging system, align with the definition of machinery classified under this HTS code. \n\nThe molds, specifically designed for injection molding of rubber parts such as O-rings and U-cups, are classified under HTS code 8480.71.8045, which covers molds for rubber or plastics, specifically

## Data Cleaning

In [7]:
from src.preprocessing import *

parsed_df = df["messages"].apply(parse_messages)

In [8]:
parsed_df['question'][0]

'What is the HTS US Code for a horizontal injection molding machine and molds for producing rubber parts?'

In [9]:
parsed_df['response'][1]

'HTS US Code -> 6111.20.1000; 6106.10.0030\nReasoning -> The product in question is a shirt constructed of knit fabric made entirely of cotton, specifically designed for infants and toddlers. The HTSUS classification is determined based on the type of garment, the material used, and the target demographic. \n\n1. The shirt is categorized under "babies garments and clothing accessories, knitted or crocheted: of cotton: blouses and shirts, except those imported as parts of sets," which aligns with HTS Code 6111.20.1000 for infant sizes (3-6 months through 18-24 months).\n   \n2. For toddler sizes (2T-3T), the shirt falls under "womens or girls blouses and shirts, knitted or crocheted: of cotton: girls other," which corresponds to HTS Code 6106.10.0030.\n\n3. Both classifications are applicable as the shirt is imported in different sizes catering to both infants and toddlers, confirming the correct HTS codes for the respective size categories.\n\n4. The duty rate for both classifications 

In [10]:
parsed_df = parsed_df.apply(
    extract_fields,
    axis=1
)

parsed_df["clean_text"] = (
    parsed_df["product_description"]
    .apply(clean_text)
)

In [11]:
parsed_df

,product_description,hts_codes,reasoning,clean_text
0,What is the HTS US Code for a horizontal injec...,8477.10.9015; 8480.71.8045,The horizontal injection molding machine is cl...,a horizontal injection molding machine and mol...
1,What is the HTS US Code for an infant and todd...,6111.20.1000; 6106.10.0030,The product in question is a shirt constructed...,an infant and toddler shirt made of 100% cotto...
2,What is the HTS US Code for L-Lysine HCL (CAS#...,2922.41.0010,The HTS US Code 2922.41.0010 is applicable to ...,l-lysine hcl (cas# 657-27-2) from china
3,What is the HTS US Code for polyvinylchloride ...,3904.10.0000,The product in question is polyvinylchloride (...,polyvinylchloride (pvc) powder used in the aut...
4,What is the HTS US Code for polyvinylchloride ...,3912.31.0090,The product in question is classified under HT...,polyvinylchloride (pvc) powder used in the man...
...,...,...,...,...
18249,"What is the HTS US Code for hot-rolled, seamle...",7304.59.8040,The products in question are identified as hol...,hot-rolled seamless steel tubular products use...
18250,What is the HTS US Code for women's knit panty...,9802.00.8044,The garments in question are classified under ...,women's knit panty and mid-thigh shaper garmen...
18251,What is the HTS US Code for Cast-Iron Cylinder...,8409.99.1040,The product in question is a Cast-Iron Cylinde...,cast-iron cylinder head castings for heavy-dut...
18252,What is the HTS US Code for a cleaning machine...,8424.30.9000,The product in question is the Ecoclean 68H HC...,a cleaning machine for tray evaporators design...


In [12]:
parsed_df.isnull().sum()

product_description    0
hts_codes              0
reasoning              0
clean_text             0
dtype: int64

In [13]:
parsed_df.duplicated().sum()

0

In [14]:
parsed_df.shape

(18254, 4)

In [15]:
parsed_df.sample(5, random_state=42)

,product_description,hts_codes,reasoning,clean_text
10731,What is the HTS US Code for a Jump N Start por...,8504.40.9530,The Jump N Start portable rechargeable power s...,a jump n start portable rechargeable power sta...
2608,What is the HTS US Code for automatic stacking...,8426.19.0000,The product in question includes three types o...,automatic stacking cranes rubber tire gantry c...
14585,"What is the HTS US Code for 1,3-Dinitrobenzene?",2904.20.3500,"The product in question is 1,3-Dinitrobenzene ...",1 3-dinitrobenzene
4254,What is the HTS US Code for a ladies handbag m...,4202.22.7000,"The product in question is a ladies handbag, s...",a ladies handbag made of silk with glass bead ...
6925,What is the HTS US Code for incomplete mini-sp...,"8415.90.8065, 8415.90.8085, 9903.88.15",The classification of the incomplete mini-spli...,incomplete mini-split air conditioning systems...


In [16]:
parsed_df["desc_length"] = (
    parsed_df["product_description"]
    .str.len()
)

parsed_df["word_count"] = (
    parsed_df["product_description"]
    .str.split()
    .str.len()
)

parsed_df[
    ["desc_length", "word_count"]
].describe()

,desc_length,word_count
count,18254.000000,18254.000000
mean,114.053303,20.129835
std,50.700849,8.287598
min,35.000000,8.000000
25%,77.000000,14.000000
50%,101.000000,18.000000
75%,139.000000,24.000000
max,410.000000,71.000000
